# Run Log Results Plotter

Use this notebook to compare VFA, hybrid, or future simulation run logs. Add one or more folders to `RUN_LOGS`; each folder is searched recursively for `results.csv`, so a parent folder containing many experiments works directly.

In [1]:
from pathlib import Path
import os
import re
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomosim_matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

WORKSPACE_ROOT = Path.cwd()
while WORKSPACE_ROOT.name != "FOMOsim" and WORKSPACE_ROOT.parent != WORKSPACE_ROOT:
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

WORKSPACE_ROOT

Matplotlib is building the font cache; this may take a moment.


PosixPath('/Users/ingvildvs/FOMOsim')

## Configuration

Edit `RUN_LOGS` and `METRICS`. By default the notebook scans every `results.csv` under `run_logs`, while skipping helper/archive folders listed in `EXCLUDED_RUN_LOG_DIRS`. You can still give exact run folders, parent folders, direct `results.csv` paths, or glob patterns such as `run_logs/run_20260509_*`.

In [2]:
BASELINE_RUN_LOGS = [
<<<<<<< local
    "run_logs/run_20260512_073645", #Hybrid Imbalance Count Squared Temporal - new version with more training and some bug fixes
    "run_logs/run_20260512_073701", #Hybrid Imbalance Squared Temporal
    "run_logs/run_20260512_084959",
    "run_logs/run_20260512_084959/hybrid_Squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_085006",
    "run_logs/run_20260512_085006/hybrid_R4_NetTemporalAlternative_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_085009",
    "run_logs/run_20260512_085009/hybrid_Imbalance_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_085012",
    "run_logs/run_20260512_085012/hybrid_Imbalance_severity_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_085028",
    "run_logs/run_20260512_085028/hybrid_Imbalance_severity_squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_131833",
    "run_logs/run_20260512_131833/vfa_Squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_133924", #VFA only mbalance Count Squared Temporal
    "run_logs/run_20260512_133924/vfa_Imbalance_squared_count_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_134012", #VFA only Imbalance Squared Temporal
    "run_logs/run_20260512_134012/vfa_Imbalance_squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_134230",
    "run_logs/run_20260512_134230/vfa_R4_NetTemporalAlternative_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_134238",
    "run_logs/run_20260512_134238/vfa_Imbalance_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_135125",
    "run_logs/run_20260512_135125/vfa_Imbalance_severity_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_135806",
    "run_logs/run_20260512_135806/vfa_Imbalance_severity_squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1",
    "run_logs/run_20260512_201228", #Imb count temp VFA
    "run_logs/run_20260512_192829", #Imb count temp VFA full
    "run_logs/run_20260512_222826", #Imb sq Hybrid
    "run_logs/run_20260512_223013", #Imb sq VFA
    "run_logs/run_20260512_223303", #Sq ony Hybird VFA
    "run_logs/run_20260513_082445" ,#Imb only Hybrid VFA
    "run_logs/run_20260514_133338",
    "run_logs/run_20260514_082001"
    
=======
     "run_logs/DN_GMP_baselines", #GMP and DN policy runs
    "run_logs/baseline_rebalancing_run_logs"
>>>>>>> remote
]

MAINTENANCE_RUN_LOGS = [
    "run_logs/DN_GMP_baselines", #GMP and DN policy runs
    "run_logs/baseline_maintenanceextension_run_logs",
]

# Choose any columns from results.csv. Run the "Available columns" cell below to inspect options.
METRICS = ["service_level", "starvations", "congestions","total_depot_visits","total_fixed_on_site","total_picked_up_to_depot","total_functional_pickups","broken_ratio_end_onsite","broken_ratio_end_depot","functional_ratio_end", "functional_ratio_start", "total_trips",]

SORT_BY = "service_level"
OUTPUT_DIR = WORKSPACE_ROOT / "run_logs" / "plots" / "baseline_candidates"
SAVE_OUTPUTS = False
SAVE_SUMMARY_TABLE_CSV = True

METRIC_LABELS = {
    "total_trips": "Total trips",
    "service_level": "Service level",
    "starvations": "Starvations",
    "congestions": "Congestions",
    "avg_bike_km_driven": "Avg. bike km driven",
    "total_depot_visits": "Depot visits",
    "total_depot_deliveries": "Depot deliveries",
    "total_fixed_on_site": "Fixed on site",
    "total_picked_up_to_depot": "Picked up to depot",
    "total_depot_fixes": "Depot fixes",
    "total_functional_pickups": "Functional pickups",
    "total_functional_deliveries": "Functional deliveries",
    "functional_ratio_start": "Functional ratio start",
    "broken_ratio_end_onsite": "End broken ratio onsite",
    "broken_ratio_end_depot": "End broken ratio depot",
    "functional_ratio_end": "End functional ratio",
}

CONSIDERED_EXPERIMENT = BASELINE_RUN_LOGS

## Helper Functions

In [3]:
def resolve_input(raw_path: str) -> list[Path]:
    raw_path = str(raw_path).rstrip(". \t")   # strip accidental trailing dots or spaces
    base_pattern = raw_path if Path(raw_path).is_absolute() else str(WORKSPACE_ROOT / raw_path)
    matches = sorted(Path(p) for p in __import__("glob").glob(base_pattern))
    if not matches:
        matches = [Path(base_pattern)]
    return matches


def path_is_excluded(candidate: Path) -> bool:
    try:
        relative_parts = candidate.relative_to(WORKSPACE_ROOT / "run_logs").parts
    except ValueError:
        relative_parts = candidate.parts
    return any(part in CONSIDERED_EXPERIMENT for part in relative_parts)


def find_results_csv(run_logs) -> list[Path]:
    # Each entry may be a plain string/path OR a list of strings to combine.
    flat: list[str] = []
    for entry in run_logs:
        if isinstance(entry, (list, tuple)):
            flat.extend(str(p) for p in entry)
        else:
            flat.append(str(entry))

    paths = []

    for raw in flat:
        for path in resolve_input(raw):
            if path_is_excluded(path):
                continue
            if path.is_file() and path.name == "results.csv":
                if not path_is_excluded(path):
                    paths.append(path)
            elif path.is_dir():
                for csv_path in sorted(path.rglob("results.csv")):
                    if not path_is_excluded(csv_path):
                        paths.append(csv_path)
            else:
                print(f"Warning: input not found, skipping: {raw}")
    return list(dict.fromkeys(paths))


def strip_known_suffixes(name: str) -> str:
    name = re.sub(r"_seed\d+.*$", "", name)
    name = re.sub(r"_TD_W\d+.*$", "", name)
    name = re.sub(r"_OS_W\d+.*$", "", name)
    name = re.sub(r"_EH_.*$", "", name)
    return name.strip("_")


def experiment_from_path(csv_path: Path) -> tuple[str, str]:
    folder = csv_path.parent.name
    if folder.startswith("hybrid_"):
        return "Hybrid", strip_known_suffixes(folder.removeprefix("hybrid_"))
    if folder.startswith("vfa_"):
        return "VFA", strip_known_suffixes(folder.removeprefix("vfa_"))
    if folder.startswith("greedy-maintenance_") or folder.startswith("GreedyMaintenance_"):
        return "Greedy", "Greedy maintenance"
    if folder.startswith("DoNothing_"):
        return "DoNothing", "Do nothing"

    try:
        first = pd.read_csv(csv_path, nrows=1)
        exp_name = str(first["exp_name"].iloc[0]) if "exp_name" in first else folder
    except Exception:
        exp_name = folder
    return "Unknown", strip_known_suffixes(exp_name)


def pretty_experiment(name: str) -> str:
    aliases = {
        "R4_NetTemporalAlternative": "Net temporal",
        "R6_BreadthDepth": "Squared count",
    }
    return aliases.get(name, name.replace("_", " "))


def load_results(run_logs: list[str]) -> pd.DataFrame:
    rows = []
    for csv_path in find_results_csv(run_logs):
        policy, experiment = experiment_from_path(csv_path)
        try:
            df = pd.read_csv(csv_path)
        except Exception as exc:
            print(f"Warning: could not read {csv_path}: {exc}")
            continue

        df["policy"] = policy
        df["experiment"] = experiment
        df["experiment_label"] = pretty_experiment(experiment)
        df["label"] = df["policy"] + ": " + df["experiment_label"]
        df["source"] = str(csv_path.relative_to(WORKSPACE_ROOT))
        rows.append(df)

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def summarize_results(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    missing = [metric for metric in metrics if metric not in df.columns]
    if missing:
        raise ValueError(f"Missing metric columns: {missing}")

    grouped = df.groupby(["policy", "experiment", "experiment_label", "label"], dropna=False)
    parts = []
    for metric in metrics:
        stats = grouped[metric].agg(mean="mean", std="std", n="count").reset_index()
        stats["metric"] = metric
        parts.append(stats)
    return pd.concat(parts, ignore_index=True)


def ordered_labels(summary: pd.DataFrame, sort_by: str | None) -> list[str]:
    if sort_by and sort_by in set(summary["metric"]):
        ranking = summary[summary["metric"] == sort_by].sort_values("mean", ascending=False)
        return ranking["label"].tolist()
    return sorted(summary["label"].unique())


def wide_summary(summary: pd.DataFrame, metrics: list[str], sort_by: str | None) -> pd.DataFrame:
    labels = ordered_labels(summary, sort_by)
    mean_wide = summary.pivot(index="label", columns="metric", values="mean").reindex(labels)
    std_wide = summary.pivot(index="label", columns="metric", values="std").reindex(labels)
    n_values = summary[summary["metric"] == metrics[0]].set_index("label")["n"].reindex(labels)

    cols = []
    for metric in metrics:
        cols.extend([(metric, "mean"), (metric, "std")])

    out = pd.concat({"mean": mean_wide, "std": std_wide}, axis=1).swaplevel(axis=1)
    out = out.reindex(columns=pd.MultiIndex.from_tuples(cols))
    out.insert(0, ("n", ""), n_values)
    return out


def plot_metric_bars(summary: pd.DataFrame, metrics: list[str], sort_by: str | None = "service_level"):
    available_metrics = [m for m in metrics if m in set(summary["metric"])]
    labels = ordered_labels(summary, sort_by)

    fig_height = max(4.0, 0.38 * len(labels) * len(available_metrics))
    fig, axes = plt.subplots(1, len(available_metrics), figsize=(5.2 * len(available_metrics), fig_height), squeeze=False)
    y = np.arange(len(labels))

    for ax, metric in zip(axes[0], available_metrics):
        metric_summary = summary[summary["metric"] == metric].set_index("label").reindex(labels).reset_index()
        means = metric_summary["mean"].to_numpy(dtype=float)
        stds = metric_summary["std"].fillna(0.0).to_numpy(dtype=float)

        ax.barh(y, means, xerr=stds, color="#4C78A8", alpha=0.9)
        ax.set_title(METRIC_LABELS.get(metric, metric.replace("_", " ").title()))
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.25)
        if metric == available_metrics[0]:
            ax.set_yticks(y, labels)
        else:
            ax.set_yticks(y, [])

        for yi, value in zip(y, means):
            if not np.isnan(value):
                txt = f"{value:.4f}" if abs(value) < 10 else f"{value:.0f}"
                ax.text(value, yi, f"  {txt}", va="center", fontsize=8)

    fig.tight_layout()
    return fig

## Load Results

In [4]:
results_csv = find_results_csv(CONSIDERED_EXPERIMENT)
print(f"Found {len(results_csv)} results.csv files")
for path in results_csv:
    print(path.relative_to(WORKSPACE_ROOT))

Found 24 results.csv files
run_logs/DN_GMP_baselines/do-nothing_TD_W34_old_D504h_V1/results.csv
run_logs/DN_GMP_baselines/greedy-maintenance_TD_W34_old_D504h_V1/results.csv
run_logs/baseline_rebalancing_run_logs/hybrid_Imbalance_count_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1/results.csv
run_logs/baseline_rebalancing_run_logs/hybrid_Imbalance_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1/results.csv
run_logs/baseline_rebalancing_run_logs/hybrid_Imbalance_severity_squared_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1/results.csv
run_logs/baseline_rebalancing_run_logs/hybrid_Imbalance_severity_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1/results.csv
run_logs/baseline_rebalancing_run_logs/hybrid_Imbalance_squared_count_temporal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1/results.csv
run_logs/baseli

In [5]:
df = load_results(CONSIDERED_EXPERIMENT)
print(f"Loaded {len(df)} seed rows from {df['label'].nunique() if not df.empty else 0} experiments")
df.head()

<<<<<<< local <removed>


Loaded 240 seed rows from 24 experiments


>>>>>>> remote <modified: >


,seed,exp_name,alpha,duration_hours,total_runtime_s,service_level,avg_bike_km_driven,starvations,congestions,total_trips,bike_departures,bike_arrivals,total_depot_deliveries,total_depot_visits,total_fixed_on_site,total_picked_up_to_depot,total_depot_fixes,total_bikes_redistributed_from_depot,total_functional_pickups,total_functional_deliveries,broken_ratio_start_onsite,broken_ratio_start_depot,functional_ratio_start,broken_ratio_end_onsite,broken_ratio_end_depot,functional_ratio_end,broken_bikes_end_onsite,broken_bikes_end_depot,new_breakdowns_onsite,new_breakdowns_depot,restored_onsite,restored_depot,policy,experiment,experiment_label,label,source
0,42,NaN,0.0,504,40320,0.8041,389.6202,9303,2506,60282,50979,50979,0,0,0,0,0,0,0,0,0.0157,0.0328,0.9514,0.2647,0.2712,0.4641,203,208,192,185,0,0,Unknown,nan,nan,Unknown: nan,run_logs/DN_GMP_baselines/do-nothing_TD_W34_ol...
1,43,NaN,0.0,504,40320,0.8183,388.2389,8502,2411,60062,51560,51560,0,0,0,0,0,0,0,0,0.0079,0.0329,0.9593,0.2520,0.2415,0.5065,193,185,188,166,0,0,Unknown,nan,nan,Unknown: nan,run_logs/DN_GMP_baselines/do-nothing_TD_W34_ol...
2,44,NaN,0.0,504,40320,0.7989,386.5737,9812,2135,59410,49598,49602,0,0,0,0,0,0,0,0,0.0157,0.0288,0.9556,0.2719,0.2928,0.4353,208,224,200,209,0,0,Unknown,nan,nan,Unknown: nan,run_logs/DN_GMP_baselines/do-nothing_TD_W34_ol...
3,45,NaN,0.0,504,40320,0.8131,386.2495,8885,2237,59520,50635,50638,0,0,0,0,0,0,0,0,0.0105,0.0276,0.9619,0.2712,0.2738,0.4550,208,210,201,197,0,0,Unknown,nan,nan,Unknown: nan,run_logs/DN_GMP_baselines/do-nothing_TD_W34_ol...
4,46,NaN,0.0,504,40320,0.8042,385.8759,9417,2121,58938,49521,49519,0,0,0,0,0,0,0,0,0.0105,0.0316,0.9578,0.2855,0.2751,0.4394,219,211,211,192,0,0,Unknown,nan,nan,Unknown: nan,run_logs/DN_GMP_baselines/do-nothing_TD_W34_ol...


## Available Columns

In [6]:
pd.DataFrame({"column": df.columns})

,column
0,seed
1,exp_name
2,alpha
3,duration_hours
4,total_runtime_s
5,service_level
6,avg_bike_km_driven
7,starvations
8,congestions
9,total_trips


## Summary Table

In [7]:
if not df.empty and df["source"].apply(lambda source: path_is_excluded(WORKSPACE_ROOT / source)).any():
    leaked = df.loc[df["source"].apply(lambda source: path_is_excluded(WORKSPACE_ROOT / source)), "source"].unique()
    leaked_text = "\n".join(leaked[:20])
    raise ValueError("Excluded run logs leaked into summary input:\n" + leaked_text)

summary = summarize_results(df, METRICS)
summary_table = wide_summary(summary, METRICS, SORT_BY)

formatters = {("n", ""): "{:.0f}"}
for metric in METRICS:
    formatters[(metric, "mean")] = "{:.4f}" if metric.endswith("ratio") or metric == "service_level" else "{:.4f}"
    formatters[(metric, "std")] = "{:.4f}" if metric.endswith("ratio") or metric == "service_level" else "{:.4f}"

summary_table.style.format(formatters).background_gradient(subset=[(SORT_BY, "mean")], cmap="Blues")

## Plot Selected Metrics

## Save Outputs

In [8]:
if SAVE_OUTPUTS or SAVE_SUMMARY_TABLE_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_SUMMARY_TABLE_CSV:
    summary_table_path = OUTPUT_DIR / "run_log_results_summary_table.csv"
    summary_table_csv = summary_table.copy()
    summary_table_csv.columns = [
        col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
        for col in summary_table_csv.columns.to_flat_index()
    ]
    summary_table_csv.to_csv(summary_table_path)
    print(f"Saved summary table: {summary_table_path.relative_to(WORKSPACE_ROOT)}")

if SAVE_OUTPUTS:
    raw_path = OUTPUT_DIR / "run_log_results_raw.csv"
    summary_path = OUTPUT_DIR / "run_log_results_summary.csv"
    figure_path = OUTPUT_DIR / "run_log_results_comparison.png"

    df.to_csv(raw_path, index=False)
    summary.to_csv(summary_path, index=False)
    fig.savefig(figure_path, dpi=220, bbox_inches="tight")

    print(f"Saved raw data: {raw_path.relative_to(WORKSPACE_ROOT)}")
    print(f"Saved summary:  {summary_path.relative_to(WORKSPACE_ROOT)}")
    print(f"Saved figure:   {figure_path.relative_to(WORKSPACE_ROOT)}")